# Enterprise RAG with Access Control — on Databricks

**Runs top-to-bottom in a Databricks notebook.** Serverless is fine.

This is the Lakehouse implementation of the design in `docs/03-theory-databricks.md`. It builds the
whole thing in your workspace and then attacks it:

| Part | What it proves |
|---|---|
| 1 | The corpus, with ABAC attributes as **Delta columns** |
| 2 | **Unity Catalog is the policy engine** — 7 rules + PII masking in a governed view |
| 3 | **Vector Search cannot even be built on a governed table** — and the two-layer answer |
| 4 | Retrieval that respects the caller: hybrid search + compiled ACL filter |
| 5 | Generation with citations, on Foundation Model APIs |
| 6 | The **zero-leak assertion**, as plain SQL |

**Case study:** Meridian Cloud, a B2B SaaS observability company. Support engineers, account managers
and security staff share one knowledge base but must see different slices of it.

> *"Why did Vertex Financial lose data in March, and do they get service credits?"* — has five
> different correct answers depending on who asks. Proving that is the point of this notebook.

**Cleanup:** the last cell drops everything it created.

> **No `%pip install` needed.** This notebook uses only the Databricks SDK, which ships with
> the runtime. Vector Search is driven through `w.api_client.do(...)` rather than the
> `databricks-vectorsearch` helper client - see the note in Part 3.

## Configuration

Edit these if you want a different catalog, schema, or endpoint.

In [1]:
CATALOG   = "main"
SCHEMA    = "meridian_rag"
TABLE     = f"{CATALOG}.{SCHEMA}.chunks"
ENTITL    = f"{CATALOG}.{SCHEMA}.user_entitlements"
SECURE    = f"{CATALOG}.{SCHEMA}.chunks_secure"   # governed view — layer 2 reads this
INDEX     = f"{CATALOG}.{SCHEMA}.chunks_idx"

# Reusing an existing endpoint avoids spinning up a new (billable) one.
# Set CREATE_ENDPOINT = True to provision your own instead.
VS_ENDPOINT      = "rag_demo_endpoint"
CREATE_ENDPOINT  = False
ENDPOINT_TYPE    = "STANDARD"          # "STORAGE_OPTIMIZED" if you provision a new one

EMBED_MODEL = "databricks-gte-large-en"
CHAT_MODEL  = "databricks-meta-llama-3-3-70b-instruct"

import json, time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
w = WorkspaceClient()                                   # notebook-native auth, no config needed
ME = spark.sql("SELECT current_user()").collect()[0][0]

print(f"running as : {ME}")
print(f"catalog    : {CATALOG}.{SCHEMA}")
print(f"vs endpoint: {VS_ENDPOINT}")

ValueError: default auth: metadata-service: Metadata Service returned empty token. Config: host=https://e2-demo-field-eng.cloud.databricks.com, account_id=e6e8162c-a42f-43a0-af86-312058795a14, workspace_id=1444828305810485, discovery_url=https://e2-demo-field-eng.cloud.databricks.com/oidc/.well-known/oauth-authorization-server, token=***, token_audience=https://e2-demo-field-eng.cloud.databricks.com/oidc/v1/token, auth_type=metadata-service, serverless_compute_id=auto, metadata_service_url=***. Env: DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_AUTH_TYPE, DATABRICKS_SERVERLESS_COMPUTE_ID, DATABRICKS_METADATA_SERVICE_URL

---
# Part 1 — The corpus, with ACL columns

The ABAC attributes live as **ordinary Delta columns** on every chunk. In production a Lakeflow
Declarative Pipeline writes these, and its real job is translating each source system's permission
model (Confluence space perms, Zendesk organisations, SharePoint groups) into this schema.

Note the group encoding: **one BOOLEAN column per group**. Vector Search filters work on scalar
columns and have no array-containment operator, so `ARRAY<STRING>` would be unfilterable.

In [ ]:
# Idempotent: start clean. NOTE - `CREATE OR REPLACE TABLE` does NOT detach an
# already-attached row filter or column mask, so a partial previous run leaves the
# table un-indexable. Dropping the schema is the reliable reset.
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

spark.sql(f"""
CREATE OR REPLACE TABLE {TABLE} (
  chunk_id        STRING NOT NULL,
  doc_id          STRING,
  title           STRING,
  content         STRING,
  -- ABAC attributes, denormalised onto every chunk
  tenant_id       STRING,
  source_system   STRING,
  sensitivity     STRING,
  sensitivity_lvl INT,             -- 0 public .. 3 restricted, so the index can do <=
  region          STRING,          -- EU | US | GLOBAL
  contains_pii    BOOLEAN,
  need_to_know    STRING,          -- compartment tag, NULL = none
  valid_from      DATE,            -- embargo
  valid_until     DATE,
  -- group membership: one BOOLEAN per group
  grp_public      BOOLEAN,
  grp_support_t1  BOOLEAN,
  grp_support_t3  BOOLEAN,
  grp_engineering BOOLEAN,
  grp_sales       BOOLEAN,
  grp_legal       BOOLEAN,
  grp_security    BOOLEAN
) TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("table created (Change Data Feed on, so Delta Sync can do incremental updates)")

In [ ]:
spark.sql(f"""
INSERT INTO {TABLE} VALUES
 ('HC-002#0','HC-002','Error MRD-5031 (Ingest Backpressure)',
  'MRD-5031 means the pipeline accepted your data but could not commit it to storage. Data receiving MRD-5031 has NOT been durably stored and must be retried. Agents 3.2+ retry automatically; older agents drop the batch silently.',
  'meridian','helpcenter','public',0,'GLOBAL',false,NULL,NULL,NULL,
  true,false,false,false,false,false,false),

 ('HC-003#0','HC-003','Rate Limits and the DPM Ceiling',
  'Every workspace has a sustained data-points-per-minute ceiling enforced by a token bucket. Exceeding it returns MRD-4290 with a Retry-After header. Enterprise ceilings are negotiated, typically 2,000,000 DPM.',
  'meridian','helpcenter','public',0,'GLOBAL',false,NULL,NULL,NULL,
  true,false,false,false,false,false,false),

 ('RB-101#0','RB-101','Runbook: Ingest Backpressure Triage',
  'In 90% of cases the bottleneck is the compaction queue, not ingest workers. Scale compaction first: meridianctl scale compaction --replicas 24. NEVER scale ingest workers first - it increases commit pressure on an already saturated storage tier and made the 14 March incident roughly 40 minutes longer.',
  'meridian','runbook','internal',1,'GLOBAL',false,NULL,NULL,NULL,
  false,false,true,true,false,false,false),

 ('TK-4471#0','TK-4471','Ticket 4471 - Vertex Financial ingestion gaps',
  'Reporter priya.raman@vertexfinancial.example. Sustained MRD-5031 between 08:50 and 10:30 UTC on 14 March. Workspace was within its contracted ceiling, so not a rate-limit issue. Customer asked about SLA credits - routed to their account manager rather than answered directly.',
  'meridian','ticket','internal',1,'EU',true,NULL,NULL,NULL,
  false,true,true,true,false,false,false),

 ('PM-2026-03-14#0','PM-2026-03-14','Post-mortem: EU Ingest Degradation, 14 March 2026',
  'Root cause: workspace ws_lmb_eu_077 added a unique request ID as a metric tag, raising active series from 90,000 to 14.2 million in twenty minutes. The cardinality explosion saturated the compaction queue and backpressure propagated region-wide. Core design flaw: no per-workspace isolation on the compaction path. Four Enterprise accounts breached their availability commitment and are credit-eligible.',
  'meridian','postmortem','confidential',2,'EU',false,NULL,NULL,NULL,
  false,false,true,true,false,false,false),

 ('CT-VTX-001#0','CT-VTX-001','Master Services Agreement - Vertex Financial',
  'Monthly availability commitment 99.9%. Service credits: below 99.9% and at or above 99.5% gives 10%; below 99.5% and at or above 99.0% gives 25%; below 99.0% gives 50%. Credits must be claimed in writing within 30 days and are the sole remedy. Annual contract value EUR 1,240,000.',
  'meridian','contract','confidential',2,'EU',false,NULL,NULL,NULL,
  false,false,false,false,true,true,false),

 ('PR-002#0','PR-002','Service Credit Approval Process',
  'Only the named account manager or legal may discuss credits with a customer. Support engineers must route all credit questions to the account manager without confirming or denying eligibility. Do not offer credits proactively.',
  'meridian','pricing','confidential',2,'GLOBAL',false,NULL,NULL,NULL,
  false,false,false,false,true,true,false),

 ('SA-2026-07#0','SA-2026-07','Security Advisory MRD-SA-2026-07 (EMBARGOED)',
  'A workspace write key could under a specific request-shaping condition write into a sibling workspace. Fixed in ingest edge 4.12.4. Customer notification must not be sent before the embargo lifts.',
  'meridian','advisory','restricted',3,'GLOBAL',false,'vuln-response',DATE'2026-09-01',NULL,
  false,false,false,false,false,false,true)
""")

display(spark.sql(f"""
  SELECT doc_id, source_system, sensitivity, region, need_to_know, valid_from,
         contains_pii
  FROM {TABLE} ORDER BY sensitivity_lvl, doc_id
"""))

---
# Part 2 — Unity Catalog is the policy engine

This is the part I had to hand-write in the platform-agnostic version. Here it is a SQL function,
and once attached it protects the table for **every** reader — this notebook, a dashboard, a job, a
`SELECT` from the SQL editor. Not just callers who go through my application code.

### The entitlements table

Real deployments join to SCIM-synced account groups via `is_account_group_member()`. An entitlements
table is used here so the notebook can demonstrate several personas without needing you to create
groups in your account console.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {ENTITL} (
  persona       STRING,
  user_email    STRING,
  role          STRING,
  tenant        STRING,
  clearance_lvl INT,
  region        STRING,
  compartment   STRING,
  is_external   BOOLEAN,
  groups        ARRAY<STRING>
)
""")

# Eight personas. Only one of them is bound to YOUR identity at a time (Part 2c);
# the rest drive the policy matrix below.
spark.sql(f"""
INSERT INTO {ENTITL} VALUES
 ('tier1',      'lena@meridian.example',  'Tier 1 Support Agent',        'meridian',1,'EU',    NULL,          false, array('grp_public','grp_support_t1')),
 ('tier3',      'marco@meridian.example', 'Tier 3 Escalation Engineer',  'meridian',2,'EU',    NULL,          false, array('grp_public','grp_support_t3','grp_engineering')),
 ('acct_mgr',   'sofia@meridian.example', 'Enterprise Account Manager',  'meridian',2,'EU',    NULL,          false, array('grp_public','grp_sales')),
 ('secops',     'ravi@meridian.example',  'Security Engineer',           'meridian',3,'GLOBAL','vuln-response',false, array('grp_public','grp_security','grp_engineering')),
 ('sec_mgr',    'erin@meridian.example',  'Security Mgr (no compartment)','meridian',3,'GLOBAL',NULL,         false, array('grp_public','grp_security')),
 ('contractor', 'tom@meridian.example',   'Contractor (external)',       'meridian',1,'US',    NULL,          true,  array('grp_public','grp_support_t1')),
 ('us_tier3',   'jin@meridian.example',   'Tier 3 Engineer (US)',        'meridian',2,'US',    NULL,          false, array('grp_public','grp_support_t3','grp_engineering')),
 -- a genuinely different tenant, holding EVERY group and top clearance
 ('other_tenant','attacker@acme.example', 'Other tenant, ALL groups',    'acme',    3,'EU',    'vuln-response',false, array('grp_public','grp_support_t1','grp_support_t3','grp_engineering','grp_sales','grp_legal','grp_security'))
""")

display(spark.sql(f"SELECT persona, role, tenant, clearance_lvl, region, compartment, is_external FROM {ENTITL}"))

### 2a. The policy, as one reviewable SQL function

Seven rules, deny-by-default. This version takes the principal's attributes **explicitly**, which
makes it testable for any persona without impersonation — and it is what drives the matrix below.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.policy_allows(
  -- principal
  p_tenant STRING, p_clearance INT, p_region STRING, p_compartment STRING,
  p_external BOOLEAN, p_groups ARRAY<STRING>,
  -- resource
  r_tenant STRING, r_sensitivity_lvl INT, r_region STRING, r_need_to_know STRING,
  r_valid_from DATE, r_valid_until DATE, r_source STRING, r_groups ARRAY<STRING>
)
RETURN
      p_tenant = r_tenant                                                    -- 1 tenant isolation
  AND r_sensitivity_lvl <= p_clearance                                       -- 2 clearance ladder
  AND (r_region = 'GLOBAL' OR r_region = p_region)                           -- 3 data residency
  AND (r_valid_from  IS NULL OR r_valid_from  <= current_date())             -- 4 embargo
  AND (r_valid_until IS NULL OR r_valid_until >= current_date())
  AND (r_need_to_know IS NULL OR r_need_to_know = p_compartment)             -- 5 need-to-know
  AND (NOT p_external OR r_source NOT IN ('contract','pricing','postmortem'))-- 6 external principals
  AND size(array_intersect(p_groups, r_groups)) > 0                          -- 7 group overlap
""")
print("policy function created")

### 2b. The visibility matrix — pure policy, no LLM anywhere

This table *is* the security specification. It can be reviewed by someone who does not read Python.

In [ ]:
matrix = spark.sql(f"""
WITH chunk_groups AS (
  SELECT doc_id, source_system, sensitivity, sensitivity_lvl, region, need_to_know,
         valid_from, valid_until, tenant_id,
         filter(array(
           CASE WHEN grp_public      THEN 'grp_public'      END,
           CASE WHEN grp_support_t1  THEN 'grp_support_t1'  END,
           CASE WHEN grp_support_t3  THEN 'grp_support_t3'  END,
           CASE WHEN grp_engineering THEN 'grp_engineering' END,
           CASE WHEN grp_sales       THEN 'grp_sales'       END,
           CASE WHEN grp_legal       THEN 'grp_legal'       END,
           CASE WHEN grp_security    THEN 'grp_security'    END
         ), x -> x IS NOT NULL) AS r_groups
  FROM {TABLE}
)
SELECT c.doc_id, c.source_system AS src, c.sensitivity,
  max(CASE WHEN e.persona='tier1'        THEN v END) AS tier1,
  max(CASE WHEN e.persona='tier3'        THEN v END) AS tier3,
  max(CASE WHEN e.persona='acct_mgr'     THEN v END) AS acct_mgr,
  max(CASE WHEN e.persona='secops'       THEN v END) AS secops,
  max(CASE WHEN e.persona='sec_mgr'      THEN v END) AS sec_mgr,
  max(CASE WHEN e.persona='contractor'   THEN v END) AS contractor,
  max(CASE WHEN e.persona='us_tier3'     THEN v END) AS us_tier3,
  max(CASE WHEN e.persona='other_tenant' THEN v END) AS other_tnt
FROM (
  SELECT c.*, e.persona,
         CASE WHEN {CATALOG}.{SCHEMA}.policy_allows(
                e.tenant, e.clearance_lvl, e.region, e.compartment, e.is_external, e.groups,
                c.tenant_id, c.sensitivity_lvl, c.region, c.need_to_know,
                c.valid_from, c.valid_until, c.source_system, c.r_groups)
              THEN 'Y' ELSE '.' END AS v
  FROM chunk_groups c CROSS JOIN {ENTITL} e
) c JOIN {ENTITL} e ON e.persona = c.persona
GROUP BY c.doc_id, c.source_system, c.sensitivity
ORDER BY c.sensitivity, c.doc_id
""")
display(matrix)

**Read the matrix.** Every `.` is a *named rule* firing:

- **tier1** can't reach the post-mortem — `clearance`
- **acct_mgr** can't reach it either — high enough clearance, but no group grants it (`default_deny`)
- **us_tier3** has the *same role and clearance as tier3* but loses the EU documents — `data_residency`
- **sec_mgr** has `restricted` clearance yet still can't read the advisory — `need_to_know`
- **contractor** is blocked from commercial sources regardless of groups — `external`
- **other_tnt** holds **every group and top clearance** and sees **nothing** — `tenant_isolation`
- Nobody sees `SA-2026-07` today — **embargoed** until 2026-09-01, evaluated with `current_date()`

---
## 2c. The constraint that decides the physical design

The obvious next move is `ALTER TABLE chunks SET ROW FILTER ...` and be done. **Databricks will not
let you** — and the reason only surfaces later, when you try to build the index:

```
BadRequest: Failed to create delta sync index main.meridian_rag.chunks_idx
  ... Table main.meridian_rag.chunks cannot have both row/column security
      and online materialized views.
```

**A Delta Sync index cannot be built on a table that has a row filter or column mask attached.**

That is a much harder constraint than *"Vector Search doesn't enforce UC row filters"*. It means the
obvious architecture is not merely weakly-governed — it is **unbuildable**. You are forced into two
physically separate objects:

```
   kb.chunks                 UNGOVERNED base table
      |                      SELECT granted ONLY to the sync service principal
      |
      +---------------->     VECTOR SEARCH INDEX      (Delta Sync reads this)
      |                      layer 1 - ACL applied by the query filter
      |
      +---------------->     kb.chunks_secure         GOVERNED dynamic view
                             SELECT granted to humans and agents
                             layer 2 - ACL + PII masking enforced by Unity Catalog
```

**The security consequence is the part to say out loud:** because the base table carries no policies,
*granting anyone `SELECT` on it bypasses the entire access model.* Locking the base table to the
pipeline identity stops being hygiene and becomes load-bearing.

A **dynamic view** carries both the row rules and the column masking in a single object, so it is the
natural fit — and here it is not one option among three, it is the only one that coexists with the
index.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {SECURE} AS
SELECT
  chunk_id, doc_id, title,
  -- column-level: the PII obligation, attached to an ALLOW
  CASE
    WHEN NOT contains_pii THEN content
    WHEN is_account_group_member('pii_readers') THEN content
    -- explicit ranges, no backslash classes: a `\w` here is eaten by the
    -- Python -> Spark SQL escaping chain and silently matches nothing.
    ELSE regexp_replace(content, '[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+[.][A-Za-z0-9.-]+', '[REDACTED_EMAIL]')
  END AS content,
  source_system, sensitivity, sensitivity_lvl, region, contains_pii
FROM {TABLE} c
-- row-level: the seven rules, evaluated against the CALLER
WHERE EXISTS (
  SELECT 1 FROM {ENTITL} e
  WHERE e.user_email = current_user()
    AND c.tenant_id = e.tenant                                       -- 1 tenant isolation
    AND c.sensitivity_lvl <= e.clearance_lvl                         -- 2 clearance ladder
    AND (c.region = 'GLOBAL' OR c.region = e.region)                 -- 3 data residency
    AND (c.valid_from  IS NULL OR c.valid_from  <= current_date())   -- 4 embargo
    AND (c.valid_until IS NULL OR c.valid_until >= current_date())
    AND (c.need_to_know IS NULL OR c.need_to_know = e.compartment)   -- 5 need-to-know
    AND (NOT e.is_external OR
         c.source_system NOT IN ('contract','pricing','postmortem')) -- 6 external principals
    AND size(array_intersect(e.groups, filter(array(                 -- 7 group overlap
          CASE WHEN c.grp_public      THEN 'grp_public'      END,
          CASE WHEN c.grp_support_t1  THEN 'grp_support_t1'  END,
          CASE WHEN c.grp_support_t3  THEN 'grp_support_t3'  END,
          CASE WHEN c.grp_engineering THEN 'grp_engineering' END,
          CASE WHEN c.grp_sales       THEN 'grp_sales'       END,
          CASE WHEN c.grp_legal       THEN 'grp_legal'       END,
          CASE WHEN c.grp_security    THEN 'grp_security'    END
        ), x -> x IS NOT NULL))) > 0
)
""")

# Bind YOUR identity to the tier1 persona so the enforcement is observable.
spark.sql(f"DELETE FROM {ENTITL} WHERE user_email = '{ME}'")
spark.sql(f"""
  INSERT INTO {ENTITL}
  SELECT 'me', '{ME}', role, tenant, clearance_lvl, region, compartment, is_external, groups
  FROM {ENTITL} WHERE persona = 'tier1'
""")

base_n   = spark.sql(f"SELECT count(*) c FROM {TABLE}").collect()[0][0]
secure_n = spark.sql(f"SELECT count(*) c FROM {SECURE}").collect()[0][0]
print("acting as a Tier-1 support agent\n")
print(f"  BASE  chunks         -> {base_n} rows   (ungoverned - pipeline identity only)")
print(f"  VIEW  chunks_secure  -> {secure_n} rows   (governed - what people read)")
display(spark.sql(f"SELECT doc_id, source_system, sensitivity FROM {SECURE} ORDER BY doc_id"))

In production this pairs with grants that make the split real:

```sql
REVOKE SELECT ON TABLE kb.chunks        FROM `analysts`;   -- nobody reads the base table
GRANT  SELECT ON VIEW  kb.chunks_secure TO   `analysts`;   -- everyone reads the view
```

### The leak assertion, as plain SQL

No agent, no LLM, no judge. This tests the **enforcement point itself** rather than my code's use
of it — which is why it is the strongest test in the design.

In [ ]:
for doc in ["CT-VTX-001", "PM-2026-03-14", "PR-002", "SA-2026-07"]:
    n = spark.sql(f"SELECT count(*) c FROM {SECURE} WHERE doc_id = '{doc}'").collect()[0][0]
    print(f"  {doc:<16} visible rows = {n}   {'PASS' if n == 0 else '*** LEAK ***'}")

### 2d. Live revocation — no reindex, no DDL

Change the entitlement (in production: a SCIM group change in Okta/Entra) and the *next query*
enforces it.

In [ ]:
spark.sql(f"""UPDATE {ENTITL}
   SET clearance_lvl = 2, groups = array('grp_public','grp_support_t3','grp_engineering')
   WHERE user_email = '{ME}'""")
print("promoted to Tier-3 (clearance=confidential, engineering groups)\n")
display(spark.sql(f"SELECT doc_id, source_system, sensitivity FROM {SECURE} ORDER BY doc_id"))

The post-mortem and the runbook appeared. **Nothing was reindexed and no DDL ran.**

The account manager's documents (`CT-VTX-001`, `PR-002`) are *still* absent — clearance went up, but
no group grants them. Clearance and group membership are independent gates.

### 2e. The PII obligation

The masking `CASE` lives inside the view, so it applies to every reader of the view.

In [ ]:
display(spark.sql(f"""
  SELECT doc_id, contains_pii, substr(content, 1, 110) AS content
  FROM {SECURE} WHERE doc_id IN ('TK-4471','HC-002') ORDER BY doc_id
"""))

Same column, same query: the ticket's email is redacted, the help-centre row is untouched —
decided by a data flag, not by the caller's code.

> This is **UC governance**, not `ai_mask`. `ai_mask` is an AI transform for content redaction; it is
> not an access-control primitive and should never be the thing standing between a user and PII.

---
# Part 3 — ⚠️ Vector Search does **not** inherit any of that

The single most important fact in this design. From the Databricks docs, verbatim:

> *"Row and column level permissions are not supported. However, you can implement your own
> application level ACLs using the filter API."*

**Why:** the index is a *derived copy*. The Delta Sync pipeline reads the source table with its own
identity and writes vectors into a serving system that has no concept of your users — there is no
query-time principal for `current_user()` to resolve against.

So the row filter protected the table. **It does not travel with the data.**

We build the index below and then prove the gap, rather than taking my word for it.

In [ ]:
# --- Vector Search helpers -------------------------------------------------
# We call the REST API through the SDK's authenticated client rather than the
# `databricks-vectorsearch` package. In this workspace the helper client's
# `create_delta_sync_index_and_wait` left the index OFFLINE, while this call
# provisions the identical index in under a minute. Fewer moving parts, and it
# inherits notebook auth for free.
def vs(method, path, body=None):
    return w.api_client.do(method, path, body=body)

def vs_search(_retries=6, **kwargs):
    """Query the index, retrying the transient not-ready state.

    A Delta Sync index that has reported ready can still answer
    `BadRequest: ... is not ready` for a while afterwards - it re-enters a
    maintenance state between syncs. Treat it as retryable rather than fatal;
    anything else propagates immediately.
    """
    for attempt in range(_retries):
        try:
            r = vs("POST", f"/api/2.0/vector-search/indexes/{INDEX}/query", kwargs)
            return (r.get("result") or {}).get("data_array") or []
        except Exception as e:
            if "not ready" in str(e).lower() and attempt < _retries - 1:
                wait = 10 * (attempt + 1)
                print(f"    index not ready, retrying in {wait}s...")
                time.sleep(wait)
                continue
            raise

if CREATE_ENDPOINT:
    try:
        vs("POST", "/api/2.0/vector-search/endpoints",
           {"name": VS_ENDPOINT, "endpoint_type": ENDPOINT_TYPE})
    except Exception as e:
        print("endpoint create:", e)
    while True:
        st = vs("GET", f"/api/2.0/vector-search/endpoints/{VS_ENDPOINT}")["endpoint_status"]["state"]
        if st == "ONLINE":
            break
        print("  waiting for endpoint...", st); time.sleep(15)

ep = vs("GET", f"/api/2.0/vector-search/endpoints/{VS_ENDPOINT}")
print(f"endpoint {VS_ENDPOINT}: {ep['endpoint_status']['state']} ({ep.get('endpoint_type')})")

In [ ]:
ACL_COLS = ["chunk_id","doc_id","title","content","tenant_id","source_system","sensitivity",
            "sensitivity_lvl","region","need_to_know","grp_public","grp_support_t1",
            "grp_support_t3","grp_engineering","grp_sales","grp_legal","grp_security"]

# Start from a clean index. Deletion is asynchronous, so pause before recreating
# under the same name - racing it is one way to end up with an OFFLINE index.
try:
    vs("DELETE", f"/api/2.0/vector-search/indexes/{INDEX}")
    print("deleted pre-existing index"); time.sleep(15)
except Exception:
    pass

vs("POST", "/api/2.0/vector-search/indexes", {
    "name": INDEX,
    "endpoint_name": VS_ENDPOINT,
    "primary_key": "chunk_id",
    "index_type": "DELTA_SYNC",
    "delta_sync_index_spec": {
        "source_table": TABLE,
        "pipeline_type": "TRIGGERED",
        "embedding_source_columns": [
            {"name": "content", "embedding_model_endpoint_name": EMBED_MODEL}],
        # Only synced columns can be filtered on or returned. The ACL columns MUST
        # be here or layer 1 has nothing to enforce with.
        "columns_to_sync": ACL_COLS,
    },
})

# Poll to ready. Treat OFFLINE/FAILED as terminal rather than waiting it out - a
# failed provision never recovers, and querying a half-built index raises a
# confusing `Vector index ... is not ready`.
last = None
for _ in range(80):
    st = vs("GET", f"/api/2.0/vector-search/indexes/{INDEX}").get("status", {})
    state = st.get("detailed_state", "?")
    if state != last:
        print(f"  {state}  rows={st.get('indexed_row_count')}"); last = state
    if st.get("ready"):
        print(f"index READY - {st.get('indexed_row_count')} rows indexed")
        break
    if any(k in str(state).upper() for k in ("OFFLINE", "FAILED")):
        raise RuntimeError(f"index provisioning failed: {state} - {st.get('message')}")
    time.sleep(15)
else:
    raise TimeoutError("index did not become ready in time")

### The proof: query the index with **no filter**

The row filter is attached to the table and I am currently a Tier-3 engineer who cannot read
contracts. Watch what the index returns anyway.

In [ ]:
rows = vs_search(
    query_text="Vertex Financial service credits and the March incident",
    columns=["chunk_id","doc_id","sensitivity","source_system"],
    num_results=10,
)

print("UNFILTERED index results:")
for r in rows:
    print(f"   {r[1]:<16} {r[3]:<12} {r[2]}")

leaked = [r[1] for r in rows if r[1] in ("CT-VTX-001","PR-002")]
print(f"\n  documents the ROW FILTER forbids me, returned by the index: {leaked}")
print("  ^ this is the gap. The index is a copy; UC row filters do not travel with it.")

---
# Part 4 — The two-layer answer

```
 ① VECTOR SEARCH FILTER  — compiled from the caller's attributes, runs as the service principal
    cheap, approximate, first line of defence
                    │  chunk_ids + ranking
                    v
 ② RE-READ FROM THE GOVERNED TABLE, AS THE USER
    Unity Catalog applies the row filter and column mask itself — THE AUTHORITY
```

### ⚠️ Layer 1 gotcha: the multi-column `OR` is *positional*

Not obvious from the docs, and it cost me two failed runs against a live index. The value must be an
**array with one element per OR clause**:

```python
{"grp_a OR grp_b": True}          # 400: "input must be an array"
{"grp_a OR grp_b": [True]}        # 400: "length of value != number of clauses"
{"grp_a OR grp_b": [True, True]}  # ✅  grp_a = true OR grp_b = true
```

Build the clause and its value array from the *same list* so they cannot drift apart. Storage-
Optimized endpoints avoid this entirely — their filters are SQL strings.

In [ ]:
def build_acl_filter(persona_row, endpoint_type=ENDPOINT_TYPE):
    """Compile a principal's attributes into a Vector Search filter (layer 1)."""
    groups = list(persona_row["groups"])
    if endpoint_type == "STORAGE_OPTIMIZED":
        clauses = [
            f"tenant_id = '{persona_row['tenant']}'",
            f"sensitivity_lvl <= {persona_row['clearance_lvl']}",
            f"region IN ('GLOBAL', '{persona_row['region']}')",
            "(" + " OR ".join(f"{g} = true" for g in groups) + ")",
        ]
        if persona_row["is_external"]:
            clauses.append("source_system NOT IN ('contract','pricing','postmortem')")
        return " AND ".join(clauses)

    # STANDARD endpoint -> dictionary filters
    f = {
        "tenant_id": persona_row["tenant"],
        "sensitivity_lvl <=": persona_row["clearance_lvl"],
        "region": ["GLOBAL", persona_row["region"]],
    }
    # positional OR: one value per clause, generated from the same list
    f[" OR ".join(groups)] = [True] * len(groups)
    if persona_row["is_external"]:
        f["source_system NOT"] = ["contract", "pricing", "postmortem"]
    return f


personas = {r["persona"]: r.asDict() for r in
            spark.sql(f"SELECT * FROM {ENTITL} WHERE persona <> 'me'").collect()}

print(json.dumps(build_acl_filter(personas["tier1"]), indent=2)
      if isinstance(build_acl_filter(personas["tier1"]), dict)
      else build_acl_filter(personas["tier1"]))

In [ ]:
def layer1(persona, question, k=10):
    """Vector Search with the compiled ACL filter + hybrid retrieval."""
    flt = build_acl_filter(personas[persona])
    rows = vs_search(
        query_text=question,
        columns=["chunk_id","doc_id","title","sensitivity","source_system"],
        num_results=k,
        query_type="HYBRID",                     # dense + BM25, managed
        filters_json=json.dumps(flt) if isinstance(flt, dict) else flt,
    )
    return [r[1] for r in rows]


Q = "Why did Vertex Financial lose data in March and do they get service credits?"

print(f'Q: "{Q}"\n')
print(f"{'persona':<14}{'layer 1 (Vector Search, ACL-filtered)'}")
print("-" * 78)
for p in ["tier1","tier3","acct_mgr","secops","contractor","us_tier3","other_tenant"]:
    print(f"{p:<14}{sorted(set(layer1(p, Q))) or 'NOTHING'}")

Every persona gets a different candidate set, and the cross-tenant principal — holding every
group and top clearance — gets **nothing at all**.

### Layer 2 — the authoritative re-read

Layer 1 ran as the service principal. Now re-read those chunk IDs **as the user**, and let Unity
Catalog decide. In a Databricks App or an agent endpoint this uses on-behalf-of-user auth; in this
notebook, *you* are the user, so a plain `SELECT` is exactly the same enforcement.

In [ ]:
def layer2(chunk_doc_ids):
    """Authoritative: re-read from the governed table. UC applies the row filter + mask."""
    if not chunk_doc_ids:
        return []
    in_list = ",".join(f"'{d}'" for d in chunk_doc_ids)
    return spark.sql(f"""
        SELECT doc_id, title, content FROM {SECURE}
        WHERE doc_id IN ({in_list})
    """).collect()


# I am currently Tier-3. Ask layer 1 with a filter that is deliberately too permissive
# (simulating a stale index or a buggy filter) and let layer 2 catch it.
over_permissive = {"tenant_id": "meridian"}          # NO acl clauses at all
rows_op = vs_search(query_text=Q, columns=["chunk_id","doc_id"],
                    num_results=10, filters_json=json.dumps(over_permissive))
candidates = sorted({r[1] for r in rows_op})
survivors  = sorted({r["doc_id"] for r in layer2(candidates)})

print(f"layer 1 returned (broken filter) : {candidates}")
print(f"layer 2 allowed  (UC decides)    : {survivors}")
print(f"DROPPED BY UNITY CATALOG         : {sorted(set(candidates) - set(survivors))}")
print("\n  ^ this is the security signal: rows the index returned that the governed")
print("    view refused. Log it and alert - it means the index is stale or the filter is wrong.")

---
# Part 5 — Generation

The model only ever sees text that came back from **layer 2**. Note there is no prompt instruction
saying "do not reveal confidential information" — that would be theatre. Unauthorised text simply
never reaches the context window.

In [ ]:
SYSTEM = """You are the Meridian Cloud support assistant. Answer ONLY from the provided passages.

Rules:
1. Use only the passages given. If they do not answer the question, say so plainly.
2. Cite every factual claim inline with its document id in square brackets, e.g. [PM-2026-03-14].
3. Never guess numbers, dates, thresholds or contractual terms - quote them exactly or omit them.
4. If a passage says a topic must be routed to another team, follow that instruction.
5. Answer the part you can and state briefly what you could not determine. Never mention that
   documents were withheld or that other material exists."""

def ask(question):
    """Full pipeline: layer 1 -> layer 2 -> generate, as the current user."""
    # layer 1 uses MY current entitlement, looked up live
    me = spark.sql(f"SELECT * FROM {ENTITL} WHERE user_email = '{ME}'").collect()[0].asDict()
    flt = build_acl_filter(me)
    rows_l1 = vs_search(query_text=question, query_type="HYBRID",
                        columns=["chunk_id","doc_id"], num_results=8,
                        filters_json=json.dumps(flt) if isinstance(flt, dict) else flt)
    candidate_docs = sorted({r[1] for r in rows_l1})

    rows = layer2(candidate_docs)                    # authoritative
    if not rows:
        return "I could not answer that from the material available to you.", []

    context = "\n\n".join(f"--- [{r['doc_id']}] {r['title']}\n{r['content']}" for r in rows)
    out = w.serving_endpoints.query(
        name=CHAT_MODEL,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=SYSTEM),
            ChatMessage(role=ChatMessageRole.USER,
                        content=f"Question:\n{question}\n\nPassages:\n{context}"),
        ],
        max_tokens=500, temperature=0.0,
    )
    return out.choices[0].message.content, [r["doc_id"] for r in rows]


text, cited = ask(Q)
print(f"[as Tier-3 engineer]\n")
print(text)
print(f"\ncontext documents: {cited}")

### The same question, as an account manager

Change the entitlement, ask again. Nothing else changes — no reindex, no redeploy.

In [ ]:
spark.sql(f"""
  UPDATE {ENTITL} SET clearance_lvl = 2, region = 'EU', is_external = false,
         groups = array('grp_public','grp_sales')
  WHERE user_email = '{ME}'
""")

text, cited = ask(Q)
print(f"[as Account Manager]\n")
print(text)
print(f"\ncontext documents: {cited}")

Two materially different answers to one question. The Tier-3 engineer gets the engineering root
cause; the account manager gets the contractual credit tiers and the routing rule — and neither can
see the other's evidence.

---
# Part 6 — The release gate

Three families of check. The third is not a metric, it is a **gate**:

| Family | Metric | Gates a release? |
|---|---|---|
| Retrieval | recall@k, MRR | no — a bug to fix next sprint |
| Generation | groundedness, refusal accuracy | no — and never gate on an LLM judge, it varies run to run |
| **Security** | **leak rate — must be exactly 0** | **yes** |

On Databricks the leak test is **set arithmetic over a SQL result** — deterministic, no judge, no
coin flip.

### ⚠️ Measure the right layer

My first version of this gate scored `layer1()` alone and reported **6 leaks** — every one of them
false. Worth understanding, because it is the same class of mistake as a false security alarm:

- `secops` and `sec_mgr` "leaked" the embargoed advisory. But **embargo and need-to-know are
  deliberately not pushed into the index filter** (§2c) — layer 1 is *supposed* to overshoot there.
- `other_tenant` "leaked" everything, because my filter builder hardcoded `tenant_id = 'meridian'`
  instead of reading the caller's tenant. A real bug, now fixed.

**Layer 1 overshooting is by design. The gate must measure what actually reaches the model** — layer 1
*then* layer 2. Below, both are reported: the gate is the final column; layer-1 overshoot is shown
beside it as the diagnostic it is.

To evaluate as each persona we rebind our own entitlement row and re-query — because the governed
view resolves `current_user()`, so this exercises the real enforcement path rather than simulating it.


In [ ]:
SECURITY_CASES = [
    ("tier1",      "What are the Vertex service credit tiers?",        ["CT-VTX-001","PR-002"]),
    ("tier1",      "What caused the March EU ingest incident?",        ["PM-2026-03-14"]),
    ("acct_mgr",   "What was the engineering root cause in March?",    ["PM-2026-03-14"]),
    ("contractor", "Show me the Vertex contract and the post-mortem.", ["CT-VTX-001","PM-2026-03-14","PR-002"]),
    ("secops",     "Any unpublished advisories on write keys?",        ["SA-2026-07"]),
    ("sec_mgr",    "Any unpublished advisories on write keys?",        ["SA-2026-07"]),
    ("us_tier3",   "What happened in the EU in March?",                ["PM-2026-03-14"]),
    ("other_tenant","Summarise every incident and contract.",          ["PM-2026-03-14","CT-VTX-001","PR-002","HC-002"]),
]


def sql_lit(v):
    return "NULL" if v is None else "'" + str(v).replace("'", "''") + "'"


def bind_as(persona):
    """Act AS a persona by rebinding our own entitlement row.

    Impersonation is not possible from a notebook, so this is how we exercise the
    real `current_user()` enforcement path for several roles.
    """
    p = personas[persona]
    groups = ",".join(sql_lit(g) for g in p["groups"])
    spark.sql(f"""
      UPDATE {ENTITL} SET
        tenant        = {sql_lit(p['tenant'])},
        clearance_lvl = {p['clearance_lvl']},
        region        = {sql_lit(p['region'])},
        compartment   = {sql_lit(p['compartment'])},
        is_external   = {str(bool(p['is_external'])).lower()},
        groups        = array({groups})
      WHERE user_email = '{ME}'
    """)


def pipeline_docs(question, k=10):
    """Return (what layer 1 proposed, what layer 2 actually allowed)."""
    me = spark.sql(f"SELECT * FROM {ENTITL} WHERE user_email = '{ME}'").collect()[0].asDict()
    flt = build_acl_filter(me)
    rows = vs_search(query_text=question, query_type="HYBRID",
                     columns=["chunk_id","doc_id"], num_results=k,
                     filters_json=json.dumps(flt) if isinstance(flt, dict) else flt)
    proposed = sorted({r[1] for r in rows})
    allowed  = sorted({r["doc_id"] for r in layer2(proposed)})
    return proposed, allowed


print(f"{'persona':<14}{'layer1 overshoot':<26}{'REACHED MODEL':<22}{'gate'}")
print("-" * 78)
total_leaks = 0
for persona, question, forbidden in SECURITY_CASES:
    bind_as(persona)
    proposed, allowed = pipeline_docs(question)
    overshoot = sorted(set(proposed) & set(forbidden))     # expected, non-gating
    leaked    = sorted(set(allowed)  & set(forbidden))     # THE GATE
    total_leaks += len(leaked)
    print(f"{persona:<14}{str(overshoot or '-'):<26}{str(leaked or 'none'):<22}"
          f"{'PASS' if not leaked else '*** LEAK ***'}")

print("-" * 78)
print(f"TOTAL LEAKS REACHING THE MODEL: {total_leaks}"
      f"   ->   {'GATE PASSED' if total_leaks == 0 else 'BLOCK THE RELEASE'}")
print()
print("Layer-1 overshoot is expected wherever the rule cannot be pushed into the index")
print("(embargo, need-to-know). That column is a staleness/diagnostic signal, not a gate.")

### And the same assertion without any agent at all

Because UC is the enforcement point, the strongest test bypasses the whole pipeline.

In [ ]:
bind_as('tier1')   # back to a Tier-1 support agent

print("as a Tier-1 agent, straight against the governed table:")
for doc in ["CT-VTX-001","PM-2026-03-14","PR-002","SA-2026-07"]:
    n = spark.sql(f"SELECT count(*) c FROM {SECURE} WHERE doc_id='{doc}'").collect()[0][0]
    print(f"   SELECT count(*) WHERE doc_id='{doc}'  ->  {n}   {'PASS' if n==0 else '*** LEAK ***'}")

---
# What to take away

1. **Vector Search does not inherit UC row filters or column masks.** The index is a derived copy;
   the filter does not travel with the data. This is the fact that shapes the architecture.
2. **Two layers.** ① a compiled ACL filter makes retrieval cheap and runs as a service principal;
   ② an on-behalf-of-user re-read from the governed table is the authority — and it is *Unity
   Catalog*, not application code.
3. **The row filter protects the table for every reader** — notebook, dashboard, job, agent.
4. **Live revocation is free** — change the entitlement (in production, a SCIM group), next query
   enforces it. No reindex.
5. **Rows the index returned that the governed view dropped** is your staleness alarm. Log it.
6. **The leak gate is deterministic SQL.** Never gate a release on an LLM judge.
7. **Fail closed.** If the governed read is unavailable, refuse — never fall back to serving layer 1's
   unfiltered results.

Next: `../INTERVIEW_SCRIPT_DATABRICKS.md` for how to present this on a whiteboard in 60 minutes.

---
## Cleanup

In [ ]:
# Drops everything this notebook created. The Vector Search ENDPOINT is left alone
# (it was pre-existing); only the index built here is deleted.
try:
    vs("DELETE", f"/api/2.0/vector-search/indexes/{INDEX}")
    print("index deleted")
except Exception as e:
    print("index delete:", e)

spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")
print(f"dropped {CATALOG}.{SCHEMA}")